# 01. KICOX 데이터 수집·통합

창원국가산단 KICOX 산업동향 5개 지표를 분기 마스터로 통합하고 품질을 검증한다.

| 항목 | 내용 |
| --- | --- |
| 분석단위 | 창원국가산단 × 업종 × 분기 |
| 기간 | 2018Q1 ~ 2026Q2 (34분기) |
| 지표 | 생산실적, 고용현황, 가동률, 입주업체, 가동업체 |
| 출처 우선순위 | KICOX 연간보정본 > 수정·재공시본 > 공공데이터포털 과거파일 |

실제 처리 로직은 `src/build_changwon_master.py`에 있고, 이 노트북은 실행과 검증만 담당한다.
원자료를 노트북에서 직접 수정하지 않는다.

## 1. 환경 및 경로

In [1]:
import os, sys, json, subprocess
import pandas as pd, numpy as np

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
PROC = os.path.join(ROOT, 'data', 'processed')
LOG  = os.path.join(ROOT, 'logs')
print('root   : repository root')
print('pandas :', pd.__version__)

# Windows에서 자식 프로세스가 stdout을 cp949로 인코딩하는 문제 방지:
# 부모가 encoding='utf-8'로 디코딩해도 자식이 cp949로 쓰면 한글이 깨질 수 있으므로
# 자식 프로세스의 출력 인코딩 자체를 utf-8로 강제한다.
env = dict(os.environ, PYTHONIOENCODING='utf-8', PYTHONUTF8='1')


root   : repository root
pandas : 3.0.5


## 2. 파이프라인 실행

`--offline`은 `data/raw`와 `data/annual_revision`만 사용한다.
원자료를 새로 내려받으려면 `--offline`을 뺀다.

In [2]:
res = subprocess.run(
    [sys.executable, os.path.join(ROOT, 'src', 'build_changwon_master.py'), '--offline'],
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace',
    env=env,
    cwd=ROOT
)

print('exit code:', res.returncode)
print(res.stdout[-4000:] if res.stdout else '')

if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:] if res.stderr else '')

exit code: 0
품질 검증

[분기] 34개 2018Q1 ~ 2026Q2 / 누락 없음
[업종] 누락 없음

[업종별 마스터 결측] (자동삭제 없음)
  production       :  10행 / 분기 1개 ['2023Q4']
  employment       :   0행 / 분기 0개 []
  op_rate_official : 250행 / 분기 25개 ['2018Q1', '2018Q2', '2018Q3', '2018Q4', '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2']...
  op_rate_approx   : 101행 / 분기 20개 ['2018Q1', '2018Q2', '2018Q3', '2018Q4', '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2']...
  firms_in         :   0행 / 분기 0개 []
  firms_op         :   0행 / 분기 0개 []

[전체 마스터 결측]
  production_total : 분기 0개 []
  employment_total : 분기 0개 []
  firms_in_total   : 분기 0개 []
  op_rate_total    : 분기 0개 []

[비공개(X)] 조사대상 5개사 이하 보호 규칙
  production: ['2023Q4']

[출처 구성]
  production : {'data_portal_monthly': 19, 'data_portal_quarterly': 3, 'kicox_annual_revision': 12}
  employment : {'data_portal_monthly': 19, 'data_portal_quarterly': 3, 'kicox_annual_revision': 12}
  op_rate    : {'data_portal_monthly': 19, 'data_portal_quarterly': 3, 'kicox_annua

## 3. 마스터 로드

In [3]:
ind = pd.read_csv(os.path.join(PROC, 'changwon_industry_master.csv'))
tot = pd.read_csv(os.path.join(PROC, 'changwon_total_master.csv'))
latest = json.load(open(os.path.join(LOG, 'latest_points.json'), encoding='utf-8'))

print(f'industry master : {len(ind)}행  ({ind.quarter.nunique()}분기 × {ind.industry.nunique()}업종)')
print(f'total master    : {len(tot)}행')
print(f'기간            : {ind.quarter.min()} ~ {ind.quarter.max()}')
print(f'latest_common_quarter = {latest["latest_common_quarter"]}')

industry master : 340행  (34분기 × 10업종)
total master    : 34행
기간            : 2018Q1 ~ 2026Q2
latest_common_quarter = 2026Q2


## 4. 결측 검증

기대값

- 업종별 production 결측 = `2023Q4`만 (연간보정본에서도 `X`, 비공개)
- 업종별 employment 결측 = 없음
- 단지 전체 production 결측 = 없음 (2023Q4 전체 생산은 존재)

`X`를 0으로 바꾸거나 보간하지 않는다.

In [4]:
print('업종별 production 결측 분기 :', sorted(ind[ind.production.isna()].quarter.unique()))
print('업종별 employment 결측 분기 :', sorted(ind[ind.employment.isna()].quarter.unique()))
print('전체   production 결측 분기 :', sorted(tot[tot.production_total.isna()].quarter.unique()))
print()
print('2023Q4 전체 생산 =', tot.loc[tot.quarter=='2023Q4','production_total'].values)
print('2024Q3 고용 출처 =', ind.loc[ind.quarter=='2024Q3','employment_source'].unique(),
      '| 전체 고용 =', tot.loc[tot.quarter=='2024Q3','employment_total'].values)

업종별 production 결측 분기 : ['2023Q4']
업종별 employment 결측 분기 : []
전체   production 결측 분기 : []

2023Q4 전체 생산 = [146848.36198411]
2024Q3 고용 출처 = <StringArray>
['kicox_annual_revision']
Length: 1, dtype: str | 전체 고용 = [120217.]


## 5. 업종별 합계와 단지 전체의 관계

생산은 업종별 합계와 단지 전체가 일치하지만(gap 0%), 고용은 일치하지 않는다.
고용 격차는 전 기간 0.46%~4.05%(중앙값 1.27%)이며 최근으로 올수록 커진다. 두 값을 혼용하지 않는다.

`prod_gap%` 최대값 100%는 업종별 생산이 전부 결측인 2023Q4에서 합계가 0으로 잡히는 것이며, 실제 불일치가 아니다.

In [5]:
chk = (ind.groupby('quarter')[['production','employment']].sum()
         .join(tot.set_index('quarter')[['production_total','employment_total']]))
chk['prod_gap%'] = (chk.production_total - chk.production) / chk.production_total * 100
chk['emp_gap%']  = (chk.employment_total - chk.employment) / chk.employment_total * 100
print(chk[['prod_gap%','emp_gap%']].describe().loc[['min','50%','max']].round(2))
chk[['prod_gap%','emp_gap%']].tail(6).round(2)

     prod_gap%  emp_gap%
min       -0.0      0.46
50%       -0.0      1.27
max      100.0      4.05


,prod_gap%,emp_gap%
quarter,,
2025Q1,0.0,3.69
2025Q2,0.0,3.63
2025Q3,-0.0,3.73
2025Q4,-0.0,3.75
2026Q1,0.0,3.97
2026Q2,0.0,4.05


## 6. YoY 검증

정의: `(당분기 / 4분기 전 - 1) × 100`

`pct_change(4, fill_method=None)`을 명시한다. pandas 2.x 기본값 `pad`는 결측을 앞 값으로 채워
기준값이 없는 분기에서도 YoY를 만들어낸다. 결측 보간은 금지한다.

기대값: 2023Q4·2024Q4 production YoY는 전 업종 NaN, 유효 분기 16개.

In [6]:
py_ = ind.pivot(index='quarter', columns='industry', values='production').sort_index()
ey_ = ind.pivot(index='quarter', columns='industry', values='employment').sort_index()
chk_p = ((py_/py_.shift(4)-1)*100).replace([np.inf,-np.inf], np.nan)
chk_e = ((ey_/ey_.shift(4)-1)*100).replace([np.inf,-np.inf], np.nan)
valid = (chk_p.notna() & chk_e.notna()).all(axis=1)
recalc = sorted(valid[valid].index)

print('독립 재계산 유효분기 :', len(recalc), '개')
print('파이프라인 결과와 일치 :', recalc == latest['yoy_valid_quarters'])
for q in ['2023Q4','2024Q4']:
    n = ind.loc[ind.quarter==q,'production_yoy'].notna().sum()
    print(f'  {q} production_yoy 유효 업종 수 = {n}/10  (기대: 0)')
print('inf 잔존 :', int(np.isinf(ind[['production_yoy','employment_yoy']].to_numpy(dtype=float)).sum()))

독립 재계산 유효분기 : 16 개
파이프라인 결과와 일치 : True
  2023Q4 production_yoy 유효 업종 수 = 0/10  (기대: 0)
  2024Q4 production_yoy 유효 업종 수 = 0/10  (기대: 0)
inf 잔존 : 0


## 7. Sensitivity 검증

기준분기의 최근 4분기와 직전 4분기, 합계 8개 분기 전체가 완전할 때만 계산한다.
`mean`/`median`은 NaN을 자동 제외하므로 집계 전에 완전성을 검사해야 한다.

기대값: 후보 5개 → 유효 3개(2025Q4, 2026Q1, 2026Q2) → 6조합, 60행.

In [7]:
sens = pd.read_csv(os.path.join(LOG, 'quadrant_sensitivity.csv'))
ends = sorted(sens.end_quarter.unique())
print('유효 기준분기 :', ends)
print('조합 수       :', len(ends)*2, '| 행수 :', len(sens))

summary = (sens.groupby('industry')
               .agg(생산_증가=('production_dir', lambda s: (s=='up').sum()),
                    고용_증가=('employment_dir', lambda s: (s=='up').sum()),
                    N=('production_dir','size')))
share = pd.read_csv(os.path.join(LOG, 'production_share.csv')).set_index('industry')
summary = share[['recent_4q_production_share','full_period_production_share']].join(summary).round(1)
summary

유효 기준분기 : ['2025Q4', '2026Q1', '2026Q2']
조합 수       : 6 | 행수 : 60


,recent_4q_production_share,full_period_production_share,생산_증가,고용_증가,N
industry,,,,,
기계,35.1,37.8,3,5,6
운송장비,29.7,25.3,5,0,6
전기전자,22.2,23.8,2,0,6
철강,11.6,11.4,6,3,6
음식료,0.9,0.9,0,0,6
석유화학,0.3,0.5,0,4,6
목재종이,0.1,0.1,6,6,6
비금속,0.1,0.1,3,0,6
기타,0.1,0.1,6,0,6


## 8. 생산비중 — 전체기간 / 최근 4분기

전체기간 평균 비중은 최근 국면 설명에 쓰지 않는다.

In [8]:
print('최근 4분기 window :', latest['window_recent'])
s = share['recent_4q_production_share']
for t in ['기계','운송장비','전기전자','철강']:
    print(f'  {t:<5} {s[t]:5.1f}%')
print(f"  운송장비+전기전자 = {s['운송장비'] + s['전기전자']:.1f}%")

최근 4분기 window : ['2025Q3', '2025Q4', '2026Q1', '2026Q2']
  기계     35.1%
  운송장비   29.7%
  전기전자   22.2%
  철강     11.6%
  운송장비+전기전자 = 52.0%


## 8-1. 전처리 품질 QA

`src/qa_master.py`는 master를 수정하지 않는 읽기 전용 검사만 수행한다.
- 구조(행수·분기·업종 340/34/10 등)와 데이터 타입, 문자열 잔재 여부
- 의도된 결측(2023Q4 production)과 `_masked`/`_source`/`_invalid_source` 일관성
- 값 범위(음수·0~100 op_rate·firms_in>=firms_op)와 단위 급변 후보 탐지(제거하지 않음)
- 업종합계 vs 단지 전체, 연간보정본 비제조 고용 대조, YoY 독립 재계산

exit code 0 = 전 항목 통과, 1 = FAIL 존재. 상세 결과는 `logs/qa_report.txt`.

In [9]:
qa = subprocess.run([sys.executable, os.path.join(ROOT,'src','qa_master.py')],
                    capture_output=True, text=True, encoding='utf-8', errors='replace',
                    env=env, cwd=ROOT)
print(qa.stdout[-5000:])
print('QA exit code:', qa.returncode, '(0 = 전 항목 통과)')

=====
B. 데이터 타입
  [PASS] industry.production numeric dtype (실제 float64)
  [PASS] industry.employment numeric dtype (실제 float64)
  [PASS] industry.op_rate_official numeric dtype (실제 float64)
  [PASS] industry.op_rate_approx numeric dtype (실제 float64)
  [PASS] industry.firms_in numeric dtype (실제 float64)
  [PASS] industry.firms_op numeric dtype (실제 float64)
  [PASS] total.production_total numeric dtype (실제 float64)
  [PASS] total.employment_total numeric dtype (실제 float64)
  [PASS] total.firms_in_total numeric dtype (실제 float64)
  [PASS] total.op_rate_total numeric dtype (실제 float64)
  [PASS] 숫자 컬럼에 문자열 잔재('X','-','null','nan') 없음 (발견: 없음)

C. 결측
  production 결측 분기: ['2023Q4']
  [PASS] 의도된 결측: industry production = 2023Q4 10행만
  [PASS] employment 결측 0 (실제 0)
  [PASS] firms_in 결측 0 (실제 0)
  [PASS] firms_op 결측 0 (실제 0)
  op_rate_official 결측 분기 (25개): ['2018Q1', '2018Q2', '2018Q3', '2018Q4', '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4']...
  op_rate_approx 

## 9. 해석 주의

- 생산과 고용의 관계는 상관이며 인과가 아니다.
- sensitivity에서 방향이 혼재하는 업종을 특정 유형으로 단정하지 않는다.
- 업종 재분류 시점(2018Q4, 2020Q3)을 가로지르는 장기 비교는 구조 변화로 해석하지 않는다.
- 가동률은 `op_rate_official`(2024Q2~, 9분기)만 핵심 분석에 사용한다.

다음 단계: `02_eda.ipynb`